# 2. Clean and Pre-process the Data

**Homework — Group 34**

This notebook continues the EDA notebook (findings cited as EDA 1.x). It repairs the data, creates features, decides how
each column is encoded and hands a ready-to-fit preprocessing pipeline to the feature-selection and modelling notebooks.

**How choices are measured.** Each option is scored on the same five time-ordered folds (with a 90-day gap) with the same
logistic regression, used only as a measuring instrument, and compared fold by fold on ROC-AUC (independent of the
decision threshold, which is tuned later). The folds share most of their training rows, so their differences are not
independent: an option is called **better** only if its mean gain exceeds two standard errors **and** it wins on all
five folds. Otherwise the current option is kept.

### 2.0 Setup

In [17]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer   # noqa: F401 (needed before importing IterativeImputer)
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TARGET = "repeat_purchase_90d"
DATA_DIR = Path("data")
OUTPUT_FILE = DATA_DIR / "preprocessed.joblib"   # read by the feature-selection and modelling notebooks

from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=FutureWarning)       # pandas/sklearn deprecation notices only
warnings.filterwarnings("ignore", category=ConvergenceWarning)  # IterativeImputer early-stopping notice (2.11)
pd.set_option("display.width", 200)

### 2.1 Load the data and sort train by date

In [18]:
train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["purchase_date"]).set_index("ID")
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["purchase_date"]).set_index("ID")
clientes_raw = pd.read_csv(DATA_DIR / "clientes.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

train_raw = train_raw.sort_values(["purchase_date", "ID"])   # ID breaks ties on the same day
print("test rows in sample_submission order:", test_raw.index.tolist() == sample["ID"].tolist())

test rows in sample_submission order: True


Since we found in EDA that all test data was from the future we decided that we should sort our training data by "purchase_date" and use "TimeSeriesSplit". ID was used to break ties between dates.

### 2.2 Empty records

In [19]:
empty = train_raw.drop(columns=["customer_id", TARGET]).isna().all(axis=1)
empty_test = test_raw.drop(columns=["customer_id"]).isna().all(axis=1)
train = train_raw.loc[~empty].copy()

print(f"empty rows removed from train: {empty.sum()} | empty rows in test: {empty_test.sum()}")
print(f"duplicated rows left in train: {train.duplicated().sum()}")
print(f"train now: {train.shape} | sorted by date: {train['purchase_date'].is_monotonic_increasing}")

empty rows removed from train: 65 | empty rows in test: 0
duplicated rows left in train: 0
train now: (6469, 38) | sorted by date: True


During EDA we discovered that 65 rows in train data only included "customer_id" and the target and nothing else, so these rows can be dropped. No such rows where found in test data.

### 2.3 Impossible values

In [ ]:
SENTINEL = 1e9

valid = train.loc[train["prior_purchase_count"] == 1]
valid = valid[valid["days_since_previous_purchase"].between(0, SENTINEL, inclusive="left")]
print(f"one prior purchase, valid days_since: {len(valid)} rows | days_since == tenure in "
      f"{(valid['days_since_previous_purchase'] == valid['customer_tenure_days']).mean():.1%} of them")

correct_values = train["prior_mean_purchase_value"].between(0, SENTINEL, inclusive="neither") & (train["prior_purchase_count"] > 0)
gap = (train.loc[correct_values, "prior_mean_purchase_value"]
       - train.loc[correct_values, "prior_total_spend"] / train.loc[correct_values, "prior_purchase_count"]).abs()
print(f"valid prior means equal to total / count (within 1 cent): {(gap < 0.01).mean():.1%}")

one prior purchase, valid days_since: 423 rows | days_since == tenure in 100.0% of them
valid prior means equal to total / count (within 1 cent): 99.2%


EDA found impossible negative and large numbers as well as what seems to be a sentinel value (1e9), these errors where also found in test data so we must use a fixed rule to apply to both. We first set the incorrect cell to missing which will be imputed in the pipeline. Two of the missing can be recovered exactly.

In [ ]:
IMPOSSIBLE_COLS = ["days_since_previous_purchase", "prior_mean_gap_days", "spend_365d"]   # negatives / sentinels (EDA 1.6)


def clean_data(df):
    """turn all impossible values to 'NAN' and then fills them with the following rules."""
    df = df.copy()
    for col in IMPOSSIBLE_COLS:
        df.loc[(df[col] < 0) | (df[col] >= SENTINEL), col] = np.nan

    # prior purchase can never exceed the total prior spend
    df.loc[df["prior_max_purchase_value"] > df["prior_total_spend"] + 0.01, "prior_max_purchase_value"] = np.nan

    # prior mean is calculated from total / count.
    count = df["prior_purchase_count"]
    df["prior_mean_purchase_value"] = df["prior_total_spend"] / count.where(count > 0)

    # with just one prior purchase, days since that purchase = customer tenure
    to_fill = (count == 1) & df["days_since_previous_purchase"].isna()
    df.loc[to_fill, "days_since_previous_purchase"] = df.loc[to_fill, "customer_tenure_days"]
    return df


def check_impossible(frames):
    """Count the impossible values in each file: negatives, sentinels and max above the total spend. To check before and after cleaning"""
    numeric = [c for c in frames["train"].select_dtypes("number").columns
               if c not in (TARGET, "customer_id", "random_noise")]
    counts = {}
    for split, df in frames.items():
        bad = ((df[numeric] < 0) | (df[numeric] >= SENTINEL)).sum()
        bad["prior_max > prior_total_spend"] = (df["prior_max_purchase_value"] > df["prior_total_spend"] + 0.01).sum()
        counts[split] = bad
    table = pd.DataFrame(counts)
    table = table[table.sum(axis=1) > 0]
    print(table.to_string() if len(table) else "no more impossible values")


print("before cleaning:")
check_impossible({"train": train, "test": test_raw})

train = clean_data(train)
test = clean_data(test_raw)

print("\nafter cleaning:")
check_impossible({"train": train, "test": test})

before cleaning:
                               train  test
days_since_previous_purchase      16     2
prior_mean_purchase_value         32     4
prior_mean_gap_days               16     2
spend_365d                        30     4
prior_max > prior_total_spend     45     6

after cleaning:
no impossible values left in train or test


### 2.4 Feature engineering
From what was found in EDA the target is driven mostly by recency and frequency, so we created features that represent these factors:

- "has_purchased_before": users who are not first time buyer, can also explain missing values in all the "prior" columns;
- "bought_30d", "bought_90d": tells us if they purchased anything recently, if not can also be a useful information;
- "purchases_per_year": all purchases made in a year, a loyal customer may be more inclined to repeat buy;
- "recent_share_30d", "recent_share_90d": share of the year's purchases that are recent. With no purchase in the last
  365 days the share is 0 by definition;
- "discount_rate": line plus global discount over the gross total, giving discounts may increase repeat purchase.

In [ ]:
NEW_FEATURES = ["has_purchased_before", "bought_30d", "bought_90d", "purchases_per_year",
                "recent_share_30d", "recent_share_90d", "discount_rate"]


def add_features(df):
    df = df.copy()
    count = df["prior_purchase_count"]
    df["has_purchased_before"] = (count >= 1).astype(int)
    df["bought_30d"] = (df["purchase_count_30d"] > 0).astype(int)
    df["bought_90d"] = (df["purchase_count_90d"] > 0).astype(int)
    df["purchases_per_year"] = count / (df["customer_tenure_days"] + 1) * 365
    df["recent_share_30d"] = (df["purchase_count_30d"] / df["purchase_count_365d"]).fillna(0)   # 0/0 -> share is 0
    df["recent_share_90d"] = (df["purchase_count_90d"] / df["purchase_count_365d"]).fillna(0)
    df["discount_rate"] = (df["Total Desc_ Linha"] + df["Total Desc_ Global"]) / df["Total Bruto"]
    return df


train = add_features(train)
test = add_features(test)

no_purchases_year = train["purchase_count_365d"] == 0
print(f"rows with no purchase in 365 days: {no_purchases_year.sum()} | "
      f"repeat rate {train.loc[no_purchases_year, TARGET].mean():.3f} vs {train[TARGET].mean():.3f} overall")
print("rows with a global discount but no line discount:",
      int(((train["Total Desc_ Global"] > 0) & (train["Total Desc_ Linha"] == 0)).sum()))
print("\nSpearman correlation with the target:")
print(train[NEW_FEATURES + ["random_noise"]].corrwith(train[TARGET], method="spearman")
      .sort_values(key=abs, ascending=False).round(3).to_string())

rows with no purchase in 365 days (share set to 0): 843 | repeat rate 0.320 vs 0.560 overall
rows with a global discount but no line discount: 37
missing / infinite values in the new features (train, test): 0 0 / 0 0

Spearman correlation with the target:
purchases_per_year      0.420
bought_90d              0.327
recent_share_90d        0.237
bought_30d              0.234
discount_rate           0.218
recent_share_30d        0.183
has_purchased_before    0.146
random_noise           -0.008



- All new features have better correlation with target than with "random_roise".
- "purchases_per_year" has the best correlation with the target, perhaps meaning that a tenured customer is more likely to repeat purchase.
- customers who have at least one purchase do not seem to have much correlation with the target, perhaps checking for more purchases makes sense.

### 2.5 Joining 'clientes.csv'

Only the three columns worth testing are joined (EDA 1.9), the others are empty or act as identifiers. We also found that "customer_since_year" could be redundant with "customer_tenure_days".

In [ ]:
CLIENT_COLS = ["country_or_region", "commercial_region", "customer_since_year"]
clients_reduced = clientes_raw.set_index("customer_id")[CLIENT_COLS]


def join_clientes(df):
    df = df.join(clients_reduced, on="customer_id")
    df["customer_age_years"] = df["purchase_date"].dt.year - df["customer_since_year"]
    return df

train = join_clientes(train)
test = join_clientes(test)

print("Spearman customer_age_years vs customer_tenure_days:",
      round(train["customer_age_years"].corr(train["customer_tenure_days"], method="spearman"), 3))

purchases dated before the customer's start year (train / test): 0 / 0
Spearman customer_age_years vs customer_tenure_days: 0.917


- We see that there is a very high correlation between our new "cutomer_age_years" and "customer_tenure_days" which was predictable and so this could be a candidate to be dropped. 

### 2.6 Inputs, target and time folds

During EDA we observed that all of the test data was in the future so we decided that a instead of using normal k-fold that would randomize all data in the fold, a TimeSeriesSplit would be more advantageous where we each training fold would be earlier than validation, and the training window would grow with each fold. 

Each training row would look 90 days into the future, so we added a 90 day gap so that if a purchase would fall inside the validation period we remove the row in order to avoid data leaks.

In [ ]:
dates_train = train["purchase_date"]
y = train[TARGET]
X_train = train.drop(columns=[TARGET, "purchase_date"])
X_test = test[X_train.columns] # same input columns as X_train whithout purchase_date
print("X_train:", X_train.shape, " X_test:", X_test.shape)


def time_splits_with_horizon(dates, n_splits=5, horizon_days=90):
    """TimeSeriesSplit on rows sorted by date, without the training rows whose 90-day label reaches the validation fold."""
    for train_idx, val_idx in TimeSeriesSplit(n_splits=n_splits).split(dates):
        val_start = dates.iloc[val_idx[0]]
        keep = dates.iloc[train_idx] < val_start - pd.Timedelta(days=horizon_days)
        yield train_idx[keep.to_numpy()], val_idx # this returns 1 fold at a time


cross_v = list(time_splits_with_horizon(dates_train))
print(pd.DataFrame([{"train rows": len(tr), "train ends": dates_train.iloc[tr].max().date(),
                     "valid starts": dates_train.iloc[va].min().date(), "valid ends": dates_train.iloc[va].max().date(),
                     "valid rows": len(va), "valid repeat rate": round(y.iloc[va].mean(), 3)}
                    for tr, va in cross_v]).to_string())

X_train: (6469, 47)  X_test: (986, 47)
   train rows  train ends valid starts  valid ends  valid rows  valid repeat rate
0         940  2017-05-12   2017-08-12  2019-01-12        1078              0.584
1        1922  2018-10-13   2019-01-12  2020-10-03        1078              0.534
2        3109  2020-07-04   2020-10-03  2022-03-05        1078              0.576
3        4189  2021-12-04   2022-03-05  2023-09-25        1078              0.520
4        5207  2023-06-24   2023-09-25  2025-02-15        1078              0.604


- "X_train" will keeps every candidate column for now.
- "purchase_date" is kept only for the folds in TimerSeriesSplit, it will bre dropped for the model.
- Each fold drops the training rows whose 90-day target window reaches into the validation period, so no training label
  is decided by purchases inside the validation fold.

### 2.7 The pipeline and the measuring tools

The pipeline has three blocks: numeric columns ('log1p', impute, scale), 0/1 flags (left as they are) and categorical
columns (impute, one-hot). 'make_preprocessor' builds it from a recipe: a dictionary of choices, so each comparison
below changes one entry. 'start_recipe' is the starting version and 'recipe' the current one. Everything fitted is refitted on the training part of each fold.

The starting recipe follows the EDA summary (1.15): identifiers, near-constant columns, 'document_series' and the
"clientes" columns are left out; "log1p" on the numeric columns (EDA 1.10); median imputation; a ""missing"" level for
categories; levels with fewer than 20 rows grouped.

In [ ]:
INSTR_C = 1.0   # strength of the measuring model; chosen in 2.8

IMPUTERS = {
    "median":     lambda: SimpleImputer(strategy="median"),
    "median+ind": lambda: SimpleImputer(strategy="median", add_indicator=True),
    "iterative":  lambda: IterativeImputer(max_iter=10, initial_strategy="median", random_state=RANDOM_STATE),
}


def make_preprocessor(recipe):
    log = [("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"))] if recipe["log"] else []
    numeric = Pipeline(log + [("impute", IMPUTERS[recipe["imputer"]]()), ("scale", StandardScaler())])
    cat_imputer = (SimpleImputer(strategy="most_frequent") if recipe["cat_fill"] == "mode"
                   else SimpleImputer(strategy="constant", fill_value="missing"))
    categorical = Pipeline([("impute", cat_imputer),
                            ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",   # unseen test levels -> rare group
                                                     min_frequency=recipe["min_freq"], sparse_output=False))])
    blocks = [("num", numeric, recipe["num"]), ("flags", "passthrough", recipe["flags"]), ("cat", categorical, recipe["cat"])]
    return ColumnTransformer([b for b in blocks if b[2]], remainder="drop")


def cv_auc(recipe, X=None):
    """ROC-AUC of the measuring model on each of the five folds."""
    X = X_train if X is None else X
    scores = []
    for tr, va in cross_v:
        model = Pipeline([("prep", make_preprocessor(recipe)),
                          ("clf", LogisticRegression(C=recipe.get("C", INSTR_C), max_iter=5000))])
        model.fit(X.iloc[tr], y.iloc[tr])
        scores.append(roc_auc_score(y.iloc[va], model.predict_proba(X.iloc[va])[:, 1]))
    return np.array(scores)


def paired_table(runs, reference):
    """Compare every run with 'reference' fold by fold and print the verdicts."""
    rows = []
    for name, s in runs.items():
        d = s - runs[reference]
        sem = d.std(ddof=1) / np.sqrt(len(d))
        wins, losses = (d > 0).sum(), (d < 0).sum()
        if name == reference:
            verdict = "reference"
        elif d.mean() > 2 * sem and wins == len(d):
            verdict = "better"
        elif -d.mean() > 2 * sem and losses == len(d):
            verdict = "worse"
        else:
            verdict = "inside the noise"
        rows.append({"version": name, "AUC": s.mean(), "dAUC": d.mean(), "SEM": sem,
                     "wins": f"{wins}/{len(d)}", "verdict": verdict})
    table = pd.DataFrame(rows).set_index("version")
    print(table.to_string(formatters={"AUC": "{:.4f}".format, "dAUC": "{:+.4f}".format, "SEM": "{:.4f}".format}))
    return table


def compare(variants, reference):
    """variants: {name: recipe} or {name: (recipe, X)}. Scores each one and compares it with 'reference'."""
    runs = {name: cv_auc(*(v if isinstance(v, tuple) else (v,))) for name, v in variants.items()}
    return paired_table(runs, reference), runs


DECISIONS = []


def decide(step, choice, evidence):
    DECISIONS.append({"step": step, "decision": choice, "evidence": evidence})
    print(f"DECISION {step}: {choice}")


def with_(recipe, **changes):
    return {**recipe, **changes}


def without(cols, *names):
    return [c for c in cols if c not in names]


FLAG_COLS = ["has_purchased_before", "bought_30d", "bought_90d"]
CAT_COLS = ["payment_type", "salesperson", "transport_method"]
LEFT_OUT = ["customer_id", "random_noise", "commercial_zone", "warehouse", "document_series",
            "country_or_region", "commercial_region", "customer_since_year", "customer_age_years"]
NUM_COLS = [c for c in X_train.select_dtypes("number").columns if c not in FLAG_COLS + LEFT_OUT]

start_recipe = {"num": NUM_COLS, "flags": FLAG_COLS, "cat": CAT_COLS,
         "log": True, "imputer": "median", "cat_fill": "constant", "min_freq": 20}
recipe = start_recipe
print(f"starting recipe: {len(NUM_COLS)} numeric, {len(FLAG_COLS)} flags, {len(CAT_COLS)} categorical columns")

starting recipe: 32 numeric, 3 flags, 3 categorical columns


### 2.8 Strength of the measuring model

The regularisation strength 'C' changes how well a logistic regression copes with many correlated columns. It is chosen
first, on the starting recipe, so that later steps do not credit a preprocessing choice with what is really a
regularisation effect.

In [26]:
C_GRID = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]   # smaller C = stronger regularisation
_, runs_C = compare({f"C={c}": with_(recipe, C=c) for c in C_GRID}, "C=1.0")

best = max(runs_C, key=lambda name: runs_C[name].mean())
print(f"\nagainst the highest mean AUC ({best}):")
_ = paired_table(runs_C, best)

           AUC    dAUC    SEM wins           verdict
version                                             
C=0.001 0.7861 +0.0122 0.0083  4/5  inside the noise
C=0.01  0.7898 +0.0159 0.0076  5/5            better
C=0.05  0.7888 +0.0149 0.0075  5/5  inside the noise
C=0.1   0.7870 +0.0131 0.0069  5/5  inside the noise
C=0.5   0.7786 +0.0046 0.0029  5/5  inside the noise
C=1.0   0.7739 +0.0000 0.0000  0/5         reference

against the highest mean AUC (C=0.01):
           AUC    dAUC    SEM wins           verdict
version                                             
C=0.001 0.7861 -0.0037 0.0011  0/5             worse
C=0.01  0.7898 +0.0000 0.0000  0/5         reference
C=0.05  0.7888 -0.0010 0.0009  2/5  inside the noise
C=0.1   0.7870 -0.0028 0.0012  1/5  inside the noise
C=0.5   0.7786 -0.0112 0.0048  0/5             worse
C=1.0   0.7739 -0.0159 0.0076  0/5             worse




- Regularisation is the largest effect in the notebook: every C up to 0.05 beats the default C = 1 on all five folds, by up to +0.016 AUC;
- C = 0.01 has the highest mean AUC. C = 0.005, 0.05 and 0.1 are inside the noise against it, C = 0.3 and 1 are worse. The data has blocks of highly correlated columns (EDA 1.13), and a strong penalty stops the model from leaning on any one of them;
- Every comparison below uses C = 0.01.



In [27]:
INSTR_C = float(best.removeprefix("C="))
decide("2.8 measuring model", f"LogisticRegression with C={INSTR_C} for every comparison below", "2.8: highest mean AUC; C >= 0.3 worse on all 5 folds")

DECISION 2.8 measuring model: LogisticRegression with C=0.01 for every comparison below


### 2.9 Columns left out at the start

Each column left out of the starting recipe is added back on its own, to check that none carries signal the model is
missing: the two identifier-like columns (EDA 1.7), the three categoricals that are near-constant or flat (EDA 1.3,
1.11) and the 'clientes' columns (joined in 2.5).

In [28]:
_ = compare({
    "current": recipe,
    "+ random_noise": with_(recipe, flags=recipe["flags"] + ["random_noise"]),   # already mean 0, sd 1: not logged
    "+ customer_id": with_(recipe, num=recipe["num"] + ["customer_id"]),
    "+ warehouse": with_(recipe, cat=recipe["cat"] + ["warehouse"]),
    "+ commercial_zone": with_(recipe, cat=recipe["cat"] + ["commercial_zone"]),
    "+ document_series": with_(recipe, cat=recipe["cat"] + ["document_series"]),
    "+ two regions (clientes)": with_(recipe, cat=recipe["cat"] + ["country_or_region", "commercial_region"]),
    "+ customer_age_years (clientes)": with_(recipe, num=recipe["num"] + ["customer_age_years"]),
}, "current")

                                   AUC    dAUC    SEM wins           verdict
version                                                                     
current                         0.7898 +0.0000 0.0000  0/5         reference
+ random_noise                  0.7894 -0.0004 0.0003  2/5  inside the noise
+ customer_id                   0.7896 -0.0002 0.0003  3/5  inside the noise
+ warehouse                     0.7902 +0.0004 0.0001  5/5            better
+ commercial_zone               0.7899 +0.0000 0.0000  3/5  inside the noise
+ document_series               0.7901 +0.0003 0.0003  3/5  inside the noise
+ two regions (clientes)        0.7913 +0.0015 0.0008  4/5  inside the noise
+ customer_age_years (clientes) 0.7893 -0.0005 0.0003  1/5  inside the noise




- 'warehouse' is the only column the folds call better (+0.0004, ahead on 5/5). It is added;
- 'random_noise' and 'customer_id' add nothing, as EDA 1.7 expected. They stay out: 'customer_id' is a label, and a model that used it would not carry over to new customers;
- 'commercial_zone' and 'document_series' are inside the noise and stay out;
- The two 'clientes' regions are the closest call (+0.0015, but ahead on only 4/5 folds), and 'customer_age_years' is slightly behind; it correlates 0.92 with 'customer_tenure_days' (2.5), so it repeats what the model already has. 'clientes.csv' is therefore not used here; the join stays in the notebook so the modelling notebook can test the regions again with a non-linear model.


In [29]:
recipe = with_(recipe, cat=recipe["cat"] + ["warehouse"])
decide("2.9 columns left out", "add warehouse; customer_id, random_noise, commercial_zone, document_series and clientes.csv stay out",
       "EDA 1.3, 1.7, 1.11 and 2.9: only warehouse is better (5/5 folds)")

DECISION 2.9 columns left out: add warehouse; customer_id, random_noise, commercial_zone, document_series and clientes.csv stay out


### 2.10 'log1p' on the numeric columns

'log1p' fails below -1, so the numeric inputs are first checked for negative values; then logging is compared with no
log at all.

In [30]:
print("negative values in the numeric inputs (train / test):",
      int((X_train[recipe["num"]] < 0).sum().sum()), "/", int((X_test[recipe["num"]] < 0).sum().sum()))
_ = compare({"log1p (current)": recipe, "no log": with_(recipe, log=False)}, "log1p (current)")

negative values in the numeric inputs (train / test): 0 / 0
                   AUC    dAUC    SEM wins    verdict
version                                              
log1p (current) 0.7902 +0.0000 0.0000  0/5  reference
no log          0.7842 -0.0059 0.0019  0/5      worse




- No numeric input is negative in train or test, so 'log1p' is safe on all of them;
- Removing the log is worse on all five folds (about -0.006 AUC), the clearest preprocessing effect in the notebook. It is kept on every numeric column, which is also simpler than choosing columns by skewness.


In [31]:
decide("2.10 log1p", "log1p on every numeric input", "EDA 1.10 and 2.10: no log is worse on 5/5 folds")

DECISION 2.10 log1p: log1p on every numeric input


### 2.11 Missing numeric values

Most missing values in the 'prior_*' columns are when the customer has too few purchases for the statistic to exist
(EDA 1.5). The table shows the share missing by number of prior purchases; then the median is compared with adding
missing-value indicators and with an iterative (regression-based) imputer.

In [32]:
print(X_train.assign(n=X_train["prior_purchase_count"].clip(upper=3))
      .groupby("n")[["prior_std_purchase_value", "prior_mean_gap_days", "prior_std_gap_days"]]
      .agg(lambda s: s.isna().mean()).round(3).rename(index={3: "3+"}).to_string())
print()
_ = compare({
    "median (current)": recipe,
    "median + indicators": with_(recipe, imputer="median+ind"),
    "iterative": with_(recipe, imputer="iterative"),
}, "median (current)")

     prior_std_purchase_value  prior_mean_gap_days  prior_std_gap_days
n                                                                     
0.0                       1.0                1.000                 1.0
1.0                       1.0                1.000                 1.0
2.0                       0.0                0.068                 1.0
3+                        0.0                0.037                 0.0

                       AUC    dAUC    SEM wins           verdict
version                                                         
median (current)    0.7902 +0.0000 0.0000  0/5         reference
median + indicators 0.7899 -0.0002 0.0003  3/5  inside the noise
iterative           0.7911 +0.0009 0.0002  5/5            better




- The table confirms the gaps are structural: 'prior_std_purchase_value' is missing for every customer with fewer than 2 prior purchases and never otherwise, and 'prior_std_gap_days' for every customer with fewer than 3. 'has_purchased_before' and 'prior_purchase_count' already tell the model when the history is short, so indicators add nothing (inside the noise);
- The **iterative imputer** is better (+0.0009, ahead on all 5 folds). It fills each column from a regression on the other columns, refitted in every training fold, so a customer's filled value follows the rest of their history instead of being one number for everyone. It is adopted.



In [33]:
recipe = with_(recipe, imputer="iterative")
decide("2.11 missing numeric values", "IterativeImputer, fitted inside each fold; no indicators", "EDA 1.5 and 2.11: iterative better on 5/5 folds; indicators inside the noise")

DECISION 2.11 missing numeric values: IterativeImputer, fitted inside each fold; no indicators


### 2.12 Missing and rare categories

A separate '"missing"' level only helps if rows with a missing category behave differently. Levels with fewer rows than
'min_frequency' are grouped into one "infrequent" column, which also receives levels never seen in train.

In [34]:
print(pd.DataFrame({c: {"levels": X_train[c].nunique(), "missing rows": int(X_train[c].isna().sum()),
                        "repeat rate if missing": round(y[X_train[c].isna()].mean(), 3)}
                    for c in recipe["cat"]}).T.to_string())
print(f"overall repeat rate: {y.mean():.3f}\n")
_ = compare({
    "'missing' level, min 20 (current)": recipe,
    "most frequent level": with_(recipe, cat_fill="mode"),
    "min_frequency 10": with_(recipe, min_freq=10),
    "min_frequency 50": with_(recipe, min_freq=50),
}, "'missing' level, min 20 (current)")

                  levels  missing rows  repeat rate if missing
payment_type         9.0         195.0                   0.564
salesperson          9.0         194.0                   0.598
transport_method    80.0         196.0                   0.551
warehouse            2.0         195.0                   0.636
overall repeat rate: 0.560

                                     AUC    dAUC    SEM wins           verdict
version                                                                       
'missing' level, min 20 (current) 0.7911 +0.0000 0.0000  0/5         reference
most frequent level               0.7910 -0.0001 0.0003  1/5  inside the noise
min_frequency 10                  0.7913 +0.0002 0.0001  4/5  inside the noise
min_frequency 50                  0.7909 -0.0002 0.0001  1/5  inside the noise



- Rows with a missing category repeat at 0.55-0.64, against 0.56 overall; the comparison shows that how they are filled changes nothing measurable (most frequent level inside the noise);
- Grouping levels below 10 or 50 rows instead of 20 is also inside the noise. The current choices stay: a '"missing"' level and 'min_frequency = 20'. Unseen test levels go to the infrequent group, so test transforms without error.



In [35]:
decide("2.12 categories", "'missing' level for gaps; levels with fewer than 20 rows grouped (unseen test levels too)",
       "EDA 1.3, missing-row repeat rates and 2.12: alternatives inside the noise")

DECISION 2.12 categories: 'missing' level for gaps; levels with fewer than 20 rows grouped (unseen test levels too)


### 2.13 Do the engineered features help?

Each engineered feature is removed in turn, and all of them together. The final say on every column belongs to the
feature-selection step; this only checks that the features are not harmful.

In [36]:
ENGINEERED = ["bought_30d", "bought_90d", "purchases_per_year", "recent_share_30d", "recent_share_90d", "discount_rate"]


def drop_cols(recipe, *names):
    return with_(recipe, num=without(recipe["num"], *names), flags=without(recipe["flags"], *names))


variants = {"with engineered features (current)": recipe, "without any": drop_cols(recipe, *ENGINEERED)}
variants.update({f"without {f}": drop_cols(recipe, f) for f in ENGINEERED})
_ = compare(variants, "with engineered features (current)")

                                      AUC    dAUC    SEM wins           verdict
version                                                                        
with engineered features (current) 0.7911 +0.0000 0.0000  0/5         reference
without any                        0.7911 -0.0000 0.0004  2/5  inside the noise
without bought_30d                 0.7911 +0.0000 0.0000  3/5  inside the noise
without bought_90d                 0.7912 +0.0001 0.0002  2/5  inside the noise
without purchases_per_year         0.7901 -0.0010 0.0005  1/5  inside the noise
without recent_share_30d           0.7918 +0.0007 0.0006  3/5  inside the noise
without recent_share_90d           0.7910 -0.0001 0.0001  2/5  inside the noise
without discount_rate              0.7912 +0.0001 0.0005  1/5  inside the noise




- None of the engineered features change the score measurably, alone or all together. Removing 'purchases_per_year' comes closest (-0.0010, behind on 4/5 folds), for the others, the model with C = 0.01 already gets most of their information from the raw counts they are built from.
- They are kept and passed to feature selection, which judges them alongside every other column. They have the highest univariate correlations with the target (2.4) and may matter more for non-linear models.


In [37]:
decide("2.13 engineered features", "keep all six; left to feature selection", "2.4 correlations and 2.13: removing them is inside the noise")

DECISION 2.13 engineered features: keep all six; left to feature selection


### 2.14 Final pipeline

The recipe is fitted on train and applied to test only as a check that both transform cleanly; the modelling notebook
refits it inside every fold.

In [38]:
preprocessor = make_preprocessor(recipe).set_output(transform="pandas")
used_cols = recipe["num"] + recipe["flags"] + recipe["cat"]
X_train_all = X_train
X_train, X_test = X_train[used_cols], X_test[used_cols]

X_check = preprocessor.fit_transform(X_train)
X_test_check = preprocessor.transform(X_test)
print("train:", X_check.shape, " test:", X_test_check.shape)
print("missing values left (train / test):", int(X_check.isna().sum().sum()), "/", int(X_test_check.isna().sum().sum()))
print("same output columns:", list(X_check.columns) == list(X_test_check.columns))
assert X_test.index.tolist() == sample["ID"].tolist(), "X_test must follow sample_submission order"
print("X_test in sample_submission order: True\n")

_ = compare({"starting recipe": (start_recipe, X_train_all), "final recipe": (recipe, X_train_all)}, "starting recipe")
print()
print(pd.DataFrame(DECISIONS).to_string(index=False))

train: (6469, 90)  test: (986, 90)
missing values left (train / test): 0 / 0
same output columns: True
X_test in sample_submission order: True

                   AUC    dAUC    SEM wins    verdict
version                                              
starting recipe 0.7898 +0.0000 0.0000  0/5  reference
final recipe    0.7911 +0.0013 0.0003  5/5     better

                       step                                                                                             decision                                                                     evidence
        2.8 measuring model                                            LogisticRegression with C=0.01 for every comparison below                         2.8: highest mean AUC; C >= 0.3 worse on all 5 folds
       2.9 columns left out add warehouse; customer_id, random_noise, commercial_zone, document_series and clientes.csv stay out             EDA 1.3, 1.7, 1.11 and 2.9: only warehouse is better (5/5 folds)
                 2.10



- **The biggest gain comes from regularisation, not from preprocessing.** Moving the measuring model from C = 1 to C = 0.01 adds about +0.016 AUC (2.8). Of the preprocessing choices, 'log1p' has the clearest effect (2.10);
- The final recipe differs from the starting one by 'warehouse' (2.9) and the iterative imputer (2.11), and is better on all five folds (+0.0013). Every other choice kept the current option because the alternatives were inside the noise;
- Fitted on all of train, the recipe produces the same columns for train and test with no missing value, and 'X_test' follows the row order of 'sample_submission.csv' (checked by an 'assert');
- Hand-off: feature selection uses C = 0.01 for its reference model and judges the engineered features; the modelling notebook tunes each model and the decision threshold.


### 2.15 Hand-off

Everything the feature-selection and modelling notebooks need is saved in one file. The preprocessor is saved
**unfitted**, so each fold refits it on its own training rows.

In [39]:
handoff = {
    "X_train": X_train,                   # cleaned rows, only the columns the recipe uses
    "y": y,                               # target, same row order as X_train
    "X_test": X_test,                     # same columns, in sample_submission order
    "dates_train": dates_train,           # purchase dates of X_train, to rebuild time splits if needed
    "cross_v": cross_v,                   # the 5 time-ordered folds with the 90-day gap
    "preprocessor": clone(preprocessor),  # the recipe, unfitted
    "column_groups": {k: recipe[k] for k in ("num", "flags", "cat")},
    "instrument_C": INSTR_C,              # C of the measuring model (2.8)
    "decisions": DECISIONS,
}
joblib.dump(handoff, OUTPUT_FILE)
print("saved to", OUTPUT_FILE)
print({name: getattr(obj, "shape", len(obj) if hasattr(obj, "__len__") else type(obj).__name__)
       for name, obj in handoff.items()})

saved to data/preprocessed.joblib
{'X_train': (6469, 39), 'y': (6469,), 'X_test': (986, 39), 'dates_train': (6469,), 'cross_v': 5, 'preprocessor': 'ColumnTransformer', 'column_groups': 3, 'instrument_C': 'float', 'decisions': 6}
